# **Import Depedencies**

In [ ]:
!pip install ultralytics
!pip install roboflow
import matplotlib.pyplot as plt
import numpy as np
import cv2

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import files
from matplotlib import pyplot as plt
from google.colab.patches import cv2_imshow

In [ ]:
import os
import albumentations as A
from glob import glob
from tqdm import tqdm

# **Collecting Dataset**

In [ ]:
rf = Roboflow(api_key="YOUR-API-KEY")
project = rf.workspace("zhaozhao-aydpp").project("clothes-classfy")
dataset = project.version(1).download("yolov8")
print("Dataset 'clothes-classfy' version 1 berhasil diunduh dalam format YOLOv8!")

In [ ]:
!ls

# **Initial Training**

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="/content/clothes-classfy-1/data.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    name="clothes-detection"
)

# **Initial Training Metrics**

In [ ]:
metrics = model.val()

mp, mr, map50, map5095 = metrics.box.mean_results()
print("Mean Precision:", mp)
print("Mean Recall:", mr)
print("mAP@0.5:", map50)
print("mAP@0.5:0.95:", map5095)

for i, name in metrics.names.items():
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f"{name} → Precision: {p:.3f}, Recall: {r:.3f}, AP50: {ap50:.3f}, AP50-95: {ap:.3f}")

In [ ]:
img_path = "/content/IMG_5795.jpg"
img = cv2.imread(img_path)

In [ ]:
def simulate_distance(image, scale):
    h, w = image.shape[:2]
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(image, (new_w, new_h))

    if scale < 1:
        canvas = np.zeros_like(image)
        xoff = (w - new_w) // 2
        yoff = (h - new_h) // 2
        canvas[yoff:yoff+new_h, xoff:xoff+new_w] = resized
        return canvas

    else:
        xoff = (new_w - w) // 2
        yoff = (new_h - h) // 2
        return resized[yoff:yoff+h, xoff:xoff+w]

In [ ]:
def adjust_brightness(image, factor):
    return np.clip(image * factor, 0, 255).astype(np.uint8)

In [ ]:
img_close  = simulate_distance(img, 1.3)
img_far    = simulate_distance(img, 0.8)
img_bright = adjust_brightness(img, 1.3)
img_dark   = adjust_brightness(img, 0.35)

generated = {
    "Jarak Dekat (Close Range)": img_close,
    "Jarak Jauh (Far Range)": img_far,
    "Cahaya Terang (Bright)": img_bright,
    "Cahaya Gelap (Dark)": img_dark
}

In [ ]:
print("=== HASIL GAMBAR SIMULASI ===\n")
for title, image in generated.items():
    print(title)
    cv2_imshow(image)
    print()

In [ ]:
model = YOLO("/content/runs/detect/clothes-detection/weights/best.pt")

generated = {
    "Jarak Dekat (Close Range)": img_close,
    "Jarak Jauh (Far Range)": img_far,
    "Cahaya Terang (Bright)": img_bright,
    "Cahaya Gelap (Dark)": img_dark
}

evaluation = {}

for name, image in generated.items():
    pred = model.predict(image, conf=0.3)
    boxes = pred[0].boxes

    labels = [model.names[int(b.cls)] for b in boxes]
    confs  = [float(b.conf) for b in boxes]
    coords = [b.xyxy[0].tolist() for b in boxes]

    evaluation[name] = {
        "Jumlah Deteksi": len(boxes),
        "Label": labels,
        "Confidence": confs,
        "BBox": coords
    }

    scale_factor = 1.5
    height, width = image.shape[:2]
    img_viz = cv2.resize(image, (int(width*scale_factor), int(height*scale_factor)))

    for b in boxes:
        cls_id = int(b.cls)
        label = model.names[cls_id]
        conf  = float(b.conf)
        x1, y1, x2, y2 = [int(coord*scale_factor) for coord in b.xyxy[0].tolist()]

        cv2.rectangle(img_viz, (x1, y1), (x2, y2), (0, 255, 0), thickness=3)
        text = f"{label} {conf:.2f}"
        cv2.putText(img_viz, text, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, fontScale=1.0, thickness=2, color=(0,255,0))

    plt.figure(figsize=(12,10))
    plt.title(name, fontsize=16)
    plt.imshow(cv2.cvtColor(img_viz, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

for name, result in evaluation.items():
    print(f"\n{name}")
    print(f"Jumlah Deteksi: {result['Jumlah Deteksi']}")
    for lbl, conf, box in zip(result['Label'], result['Confidence'], result['BBox']):
        print(f"Label: {lbl}, Confidence: {conf:.2f}, BBox: {box}")

In [ ]:
conditions = []
jumlah_deteksi = []
mean_confidence = []

for name, info in evaluation.items():
    conditions.append(name)
    jumlah_deteksi.append(info["Jumlah Deteksi"])

    if len(info["Confidence"]) > 0:
        mean_confidence.append(np.mean(info["Confidence"]))
    else:
        mean_confidence.append(0)

plt.figure(figsize=(8,5))
plt.bar(conditions, jumlah_deteksi)
plt.title("Perbandingan Jumlah Deteksi YOLO per Kondisi")
plt.ylabel("Jumlah Deteksi")
plt.xlabel("Kondisi Gambar")
plt.xticks(rotation=20)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(conditions, mean_confidence)
plt.title("Rata-Rata Confidence YOLO per Kondisi")
plt.ylabel("Confidence (0–1)")
plt.xlabel("Kondisi Gambar")
plt.xticks(rotation=20)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()

In [ ]:
train_images_path = "/content/train/images"
train_labels_path = "/content/train/labels"

aug_train_images = "/content/train_aug/images"
aug_train_labels = "/content/train_aug/labels"
os.makedirs(aug_train_images, exist_ok=True)
os.makedirs(aug_train_labels, exist_ok=True)

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

image_files = glob(os.path.join(train_images_path, "*.jpg"))
print(f"Mulai augmentasi {len(image_files)} gambar...\n")

for img_file in tqdm(image_files, desc="Augmenting"):
    filename = os.path.basename(img_file)
    label_file = os.path.join(train_labels_path, filename.replace(".jpg", ".txt"))

    image = cv2.imread(img_file)
    if image is None:
        print(f"Gagal membaca {filename}")
        continue

    bboxes = []
    class_labels = []
    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                x_center, y_center, w, h = map(float, parts[1:5])
                bboxes.append([x_center, y_center, w, h])
                class_labels.append(cls)
    else:
        print(f"Label tidak ditemukan untuk {filename}")
        continue

    augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
    image_aug = augmented['image']
    bboxes_aug = augmented['bboxes']
    class_labels_aug = augmented['class_labels']

    out_img_file = os.path.join(aug_train_images, filename)
    cv2.imwrite(out_img_file, image_aug)

    out_label_file = os.path.join(aug_train_labels, filename.replace(".jpg", ".txt"))
    with open(out_label_file, 'w') as f:
        for cls, bbox in zip(class_labels_aug, bboxes_aug):
            f.write(f"{cls} {' '.join(map(str, bbox))}\n")

    print(f"{filename} berhasil diaugmentasi ({len(bboxes_aug)} bbox)")

print("\nAugmentasi selesai!")

In [ ]:
results = model.predict(source="orangjeans2.png", save=True, conf=0.5)

# **Full Training + Valid Metrics**

In [ ]:
full_results = model.train(
    data=dataset.location + "/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    name="ppe-detection-full"
)

In [ ]:
best_model = "runs/detect/ppe-detection-full/weights/best.pt"
print(f"\nModel terbaik disimpan di: {best_model}")

In [ ]:
model = YOLO(best_model)

def detect_apd(video_path):
    results = model.predict(video_path, save=True, conf=0.25)
    return results

# **Full Training Test Metrics**

In [ ]:
model = YOLO("runs/detect/ppe-detection-full/weights/best.pt")

metrics = model.val(split="test")

print(metrics)
print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

In [ ]:
!pip freeze > requirements.txt

In [ ]:
files.download("requirements.txt")